### RF Model Training
This noteboook contains the workflow for training the capital and labor intensity random forest models. First, a diagnostic model is trained for model evaluation using spatial CV. Then, a final model is trained using all available data. The output is a set of trained RF models.

In [7]:
##### NB: 
# Part 1 takes ~50  min to run

In [8]:
from pathlib import Path
import json
import dataclasses
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from quantile_forest import RandomForestQuantileRegressor
import joblib

from step_a_config import RunConfig
from step_b_spatial_CV import make_spatial_folds
from step_c_train import train_model, compute_sample_weights, predict
from step_d_feature_importance import get_feature_importance

In [9]:
##### SET-UP DIRECTORIES 

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models_final/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

#### Part 1: Train diagnostic models

In [10]:
##### FEATURE COLUMNS
# feature columns were selected using a Recursive Feature Elimination (RFE) approach
# these feature lists were the outcome of that approach 

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD']

labor_cols = ['rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_ruminants_share_base_100_production_USD']

In [11]:
##### DEFINE RUNS

RUNS = [
    RunConfig(
        run_name         = 'capital_rf_spatial_CV',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',   
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "capital_folds.csv",
        model_type       = 'rf',
        feature_cols     = capital_cols, 
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'capital_qrf_spatial_CV',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',   
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "capital_folds.csv",
        model_type       = 'qrf',
        feature_cols     = capital_cols, 
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'labor_rf_spatial_CV',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',   
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "labor_folds.csv",
        model_type       = 'rf',
        feature_cols     = labor_cols, 
        version          = 'labor',
    ),
    RunConfig(
        run_name         = 'labor_qrf_spatial_CV',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',   
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "labor_folds.csv",
        model_type       = 'qrf',
        feature_cols     = labor_cols, 
        version          = 'labor',
    ),
]

In [12]:
##### RUN   

# function to save results to RESULTS_DIR
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    config_dict = dataclasses.asdict(config)
    config_dict = {k: str(v) if isinstance(v, Path) else v for k, v in config_dict.items()}
    (out_dir / "config.json").write_text(json.dumps(config_dict, indent=2))

    # combined test + train predictions with fold and split columns
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # best hyperparameters per fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # feature importance 
    results["importance"].to_csv(out_dir / "feature_importance.csv", index=False)

# wrapper function for runs
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"{'═'*60}")

    df = pd.read_csv(DATA_DIR / config.dataset)

    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, folds, config)

    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)
    print ("Saved successfully!")

# run model training
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: capital_rf_spatial_CV
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
  fold_1: 18 train countries (1,873 rows) | 1 test countries (1,037 rows)
  fold_2: 18 train countries (1,849 rows) | 1 test countries (1,061 rows)
  fold_3: 18 train countries (2,636 rows) | 1 test countries (274 rows)
  fold_4: 15 train countries (2,640 rows) | 4 test countries (270 rows)
  fold_5: 7 train countries (2,642 rows) | 12 test countries (268 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

── fold_2 ──────────────────────────────
  test countries: ['USA']
  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': None}

── fold_3 ──────────────────

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

── fold_4 ──────────────────────────────
  test countries: ['BOL', 'BGR', 'CAN', 'GRC', 'JPN', 'NZL', 'RUS', 'ESP', 'TUR', 'QAT']


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': None}

── fold_5 ──────────────────────────────
  test countries: ['ARG', 'HRV', 'GHA', 'ITA', 'MEX', 'NGA', 'POL', 'PRT', 'ZAF', 'GBR', 'VNM']
  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

── Feature importance ───────────────────────────────────

── Saving ───────────────────────────────────────────────
Saved successfully!


#### Part 2: Train full models

In [13]:
##### HYPERPARAMETERS
# hyperparameters selected manually based on most common params across folds in diagnostic runs
BEST_PARAMS_CAPITAL_RF  = {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}
BEST_PARAMS_CAPITAL_QRF = {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}
BEST_PARAMS_LABOR_RF    = {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}
BEST_PARAMS_LABOR_QRF   = {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

BEST_PARAMS_LOOKUP = {
    'capital_rf_final':  BEST_PARAMS_CAPITAL_RF,
    'capital_qrf_final': BEST_PARAMS_CAPITAL_QRF,
    'labor_rf_final':    BEST_PARAMS_LABOR_RF,
    'labor_qrf_final':   BEST_PARAMS_LABOR_QRF,
}

In [14]:
##### DEFINE RUNS

FINAL_RUNS = [
    RunConfig(
        run_name         = 'capital_rf_final',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = 'none',
        model_type       = 'rf',
        feature_cols     = capital_cols,
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'capital_qrf_final',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = 'none',
        model_type       = 'qrf',
        feature_cols     = capital_cols,
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'labor_rf_final',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = 'none',
        model_type       = 'rf',
        feature_cols     = labor_cols,
        version          = 'labor',
    ),
    RunConfig(
        run_name         = 'labor_qrf_final',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = 'none',
        model_type       = 'qrf',
        feature_cols     = labor_cols,
        version          = 'labor',
    ),
]

In [15]:
##### RUN

# function to train model on full dataset with fixed hyperparameters 
def train_final_model(df, config, best_params):
    
    X = df[config.feature_cols]
    y = df[config.target]

    # initialize model with best hyperparameters 
    if config.model_type == "rf":
        model = RandomForestRegressor(
            **best_params,
            random_state = config.random_seed,
            n_jobs       = -1,
        )
    elif config.model_type == "qrf":
        model = RandomForestQuantileRegressor(
            **best_params,
            random_state = config.random_seed,
            n_jobs       = -1,
        )

    # compute sample weights on full dataset
    sample_weight = compute_sample_weights(df["country_ID"], config.weighting)

    print(f"  fitting on {len(df):,} rows with {len(config.feature_cols)} features...")
    model.fit(X, y, sample_weight=sample_weight)
    print(f"  done")

    # predict on full dataset
    preds = predict(model, X, config)
    if isinstance(preds, pd.Series):
        preds = preds.to_frame()

    # build predictions df
    pred_df = df[config.id_cols + [config.target]].copy().reset_index(drop=True)
    pred_df = pd.concat([pred_df, preds.reset_index(drop=True)], axis=1)

    return model, pred_df

# function to save results 
def save_final_results(model, pred_df, config, best_params, out_dir):

    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    config_dict = dataclasses.asdict(config)
    config_dict = {k: str(v) if isinstance(v, Path) else v for k, v in config_dict.items()}
    config_dict["best_params"] = best_params  
    (out_dir / "config.json").write_text(json.dumps(config_dict, indent=2))

    # model
    joblib.dump(model, out_dir / "model.joblib")

    # predictions
    pred_df.to_parquet(out_dir / "predictions.parquet", index=False)
    print ("Saved successfully!")

# wrapper function for runs
def run_final(config):
    print(f"\n{'═'*60}")
    print(f"  FINAL RUN: {config.run_name}")
    print(f"{'═'*60}")

    df = pd.read_csv(DATA_DIR / config.dataset)
    best_params = BEST_PARAMS_LOOKUP[config.run_name]
    print(f"  best params: {best_params}")

    print("\n── Training ─────────────────────────────────────────────")
    model, pred_df = train_final_model(df, config, best_params)

    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_final_results(model, pred_df, config, best_params, out_dir)
    
# run model 
for config in FINAL_RUNS:
    run_final(config)


════════════════════════════════════════════════════════════
  FINAL RUN: capital_rf_final
════════════════════════════════════════════════════════════
  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

── Training ─────────────────────────────────────────────
  fitting on 2,910 rows with 11 features...
  done

── Saving ───────────────────────────────────────────────
Saved successfully!

════════════════════════════════════════════════════════════
  FINAL RUN: capital_qrf_final
════════════════════════════════════════════════════════════
  best params: {'n_estimators': 2000, 'min_samples_leaf': 5, 'max_features': 0.75, 'max_depth': 20}

── Training ─────────────────────────────────────────────
  fitting on 2,910 rows with 11 features...
  done

── Saving ───────────────────────────────────────────────
Saved successfully!

════════════════════════════════════════════════════════════
  FINAL RUN: labor_rf_final
═════════════════════════